Tentukan Tema Analisis
Lingkungan 
Energi 
Climate 
Dll

Gunakan Minimal 2 Variabel
Sesuaikan dengan kebutuhan dan tujuan analisis

Gunakan Format GeoJson
Hal ini untuk mempermudah upload di platform 
GEOMAPID yang akan digunakan url API nya

Gunakan Buat Acuan Kolom Skoring
Buat atribut/kolom yang akan dijadikan acuan untuk 
menentuan skor hingga overlay 

Overlay Intersect
Lakukan Intersect data/variabel yang sudah 
digunakan

## Tema Analisis: Pariwisata

Pada Task 6, Task 7, dan pengembangan hingga final project, saya mengangkat sebuah ide utama yaitu **Bandung Hiking Trails Explorer**.

Bandung Hiking Trails Explorer adalah sebuah website yang dirancang untuk membantu user mengeksplorasi jalur hiking di **Bandung dan sekitarnya**. Website ini menampilkan informasi penting mengenai setiap trail, yaitu **panjang rute**, **elevation gain**, **estimasi durasi hiking**, serta **aksesibilitas menuju titik awal trail** dari **stasiun** atau **halte terdekat**. Dengan demikian, user dapat memperoleh gambaran yang lebih jelas untuk membantu perencanaan perjalanan hiking mereka.

Target user dari website ini adalah **hiker** atau calon pengunjung yang ingin mencari jalur hiking yang sesuai dengan kemampuan, waktu tempuh, dan kemudahan akses transportasi.

Dalam project ini, saya menggunakan beberapa variabel utama, yaitu:

- **Data line rute trail**, yang memuat informasi geometri jalur hiking
- **Panjang trail**
- **Elevation gain**
- **Estimasi durasi hiking**
- **Data point halte dan stasiun** di Bandung

Berdasarkan variabel-variabel tersebut, saya akan melakukan pengolahan dan skoring untuk menghasilkan sebuah **kolom difficulty** yang merepresentasikan tingkat kesulitan tiap hiking trail. Skor ini diharapkan dapat membantu pengguna membandingkan trail secara lebih cepat dan intuitif.

Sebagai luaran, project ini akan dikembangkan menjadi sebuah WebGIS yang memungkinkan pengguna melihat trail dalam dua tingkat detail:  
- pada tampilan **zoom jauh**, trail ditampilkan sebagai **titik ringkasan** agar peta tetap mudah dibaca  
- pada tampilan **zoom dekat**, trail ditampilkan sebagai **jalur detail** agar pengguna dapat melihat bentuk rute secara lebih jelas  

Setiap trail juga akan dilengkapi dengan informasi interaktif yang dapat ditampilkan saat pengguna memilih trail, seperti **nama trail**, **difficulty**, **estimasi durasi**, **elevation gain**, dan informasi aksesibilitasnya.

In [1]:
import geopandas as gpd
import folium
import pandas as pd
import numpy as np
from shapely.geometry import Point

Step 1: Generate jarak setiap trail menggunakan shapely

In [38]:
trails = gpd.read_file("../Data/GeoJSON/Merged_Trail.geojson")
trails.head()

print(trails.columns)
print(trails.crs)
print(len(trails))

trails["distance_km"] = trails.geometry.length / 1000
trails[["Name", "distance_km"]].head()

print(trails)

Index(['Name', 'id', 'Duration_h', 'Elevation Gain_m', 'geometry'], dtype='object')
EPSG:32748
15
                               Name  id  Duration_h  Elevation Gain_m  \
0              Batujajar - Saguling   1         5.0                79   
1               Cikole - Parongpong   3         4.5               367   
2                Cililin - Saguling   4         5.0               276   
3             Citatah - Parahyangan   5         4.0               287   
4                        Papandayan  12         4.0               687   
5                  Lembang - Cartil   7         8.5               556   
6            Manglayang Exploration  10         9.0               926   
7   Burangrang via Tanjakan Mentari   2         6.0               932   
8                           Malabar   8         2.5               521   
9      Manglayang - Batu Kuda Trail   9         3.5               678   
10                      Pangradinan  11         2.0               215   
11                Situ Gun

Step 2: Generate point start dari array pertama line layer trail

In [37]:
def get_start_point(geom):
    if geom.geom_type == "LineString":
        return Point(geom.coords[0])
    elif geom.geom_type == "MultiLineString":
        first_line = list(geom.geoms)[0]
        return Point(first_line.coords[0])
    return None
trails_point = trails.copy()
trails_point["geometry"] = trails_point.geometry.apply(get_start_point)
print(trails_point)

                               Name  id  Duration_h  Elevation Gain_m  \
0              Batujajar - Saguling   1         5.0                79   
1               Cikole - Parongpong   3         4.5               367   
2                Cililin - Saguling   4         5.0               276   
3             Citatah - Parahyangan   5         4.0               287   
4                        Papandayan  12         4.0               687   
5                  Lembang - Cartil   7         8.5               556   
6            Manglayang Exploration  10         9.0               926   
7   Burangrang via Tanjakan Mentari   2         6.0               932   
8                           Malabar   8         2.5               521   
9      Manglayang - Batu Kuda Trail   9         3.5               678   
10                      Pangradinan  11         2.0               215   
11                Situ Gunung Trail  13         7.5               535   
12        Lembah Tengkorak via Kina   6         2.0

Step 3: Import data stasiun dan halte di Bandung (Sumber: MAPID)

In [ ]:
stasiun_bdg = gpd.read_file("../Data/GeoJSON/Stasiun_Bandung.geojson").to_crs(epsg=32748)
stasiun_kbb = gpd.read_file("../Data/GeoJSON/Stasiun_KBB.geojson").to_crs(epsg=32748)
stasiun_kb = gpd.read_file("../Data/GeoJSON/Stasiun_KB.geojson").to_crs(epsg=32748)
stasiun_purwakarta = gpd.read_file("../Data/GeoJSON/Stasiun_Purwakarta.geojson").to_crs(epsg=32748)
stasiun_garut = gpd.read_file("../Data/GeoJSON/Stasiun_Garut.geojson").to_crs(epsg=32748)
    
print(stasiun.columns)

Index(['NAMA', 'TIPE_1', 'TIPE_2', 'TIPE_3', 'ALAMAT', 'ID_PROV', 'PROVINSI',
       'ID_KABKOT', 'KABKOT', 'ID_KEC', 'KECAMATAN', 'ID_DESA', 'DESA',
       'LONGITUDE', 'LATITUDE', 'WAKTU', 'geometry'],
      dtype='object')


Step 4: Generate kolom jarak untuk jarak point start dari stasiun terdekat

In [18]:
nearest = gpd.sjoin_nearest(
    trails_point,
    stasiun,
    how="left",
    distance_col="access_dist_m"
)

nearest["access_dist_km"] = nearest["access_dist_m"] / 1000

nearest = nearest.rename(columns={"NAMA": "nearest_stop"})

Step 5: Scoring system dari elevation gain dan duration

In [50]:
nearest["score_duration"] = pd.qcut(
    nearest["Duration_h"], q=3, labels=[1, 2, 3]
).astype(int)

nearest["score_elev"] = pd.qcut(
    nearest["Elevation Gain_m"], q=3, labels=[1, 2, 3]
).astype(int)

nearest["effort_score"] = nearest["score_duration"] + nearest["score_elev"]

def classify_difficulty(x):
    if x <= 2:
        return "Easy"
    elif x <= 4:
        return "Moderate"
    else:
        return "Hard"

nearest["difficulty"] = nearest["effort_score"].apply(classify_difficulty)

nearest.head()

print(len(nearest))
print(nearest["id"].nunique())

nearest = nearest.sort_values(["id", "access_dist_m", "index_right"]) \
                 .drop_duplicates(subset="id", keep="first") \
                 .reset_index(drop=True)

nearest["id"].value_counts().head(10)

15
15


id
1     1
2     1
3     1
4     1
5     1
6     1
7     1
8     1
9     1
10    1
Name: count, dtype: int64

In [51]:
trails_final = trails.merge(
    nearest[[
        "id",
        "nearest_stop",
        "access_dist_km",
        "score_duration",
        "score_elev",
        "effort_score",
        "difficulty"
    ]],
    on="id",
    how="left"
)

trails_final.head()

,Name,id,Duration_h,Elevation Gain_m,geometry,distance_km,nearest_stop,access_dist_km,score_duration,score_elev,effort_score,difficulty
0,Batujajar - Saguling,1,5.0,79,MULTILINESTRING Z ((775298.785 9234239.272 656...,12.610050,STASIUN CIMINDI,8.216166,2,1,3,Moderate
1,Cikole - Parongpong,3,4.5,367,MULTILINESTRING Z ((793097.106 9249265.548 131...,11.891014,STASIUN CIKUDAPATEUH,15.105966,2,2,4,Moderate
2,Cililin - Saguling,4,5.0,276,MULTILINESTRING Z ((771726.661 9230328.961 700...,9.643058,STASIUN CIMINDI,13.128927,2,1,3,Moderate
3,Citatah - Parahyangan,5,4.0,287,MULTILINESTRING Z ((772374.638 9244082.033 731...,9.945255,STASIUN CIMINDI,12.810449,2,1,3,Moderate
4,Papandayan,12,4.0,687,MULTILINESTRING Z ((801941.063 9190666.289 209...,7.167687,STASIUN GEDEBAGE,41.537121,2,3,5,Hard


In [53]:
points_final = nearest[[
    "id",
    "Name",
    "Duration_h",
    "Elevation Gain_m",
    "distance_km",
    "nearest_stop",
    "access_dist_km",
    "effort_score",
    "difficulty",
    "geometry"
]].copy()

trails_final = trails_final[[
    "id",
    "Name",
    "Duration_h",
    "Elevation Gain_m",
    "distance_km",
    "nearest_stop",
    "access_dist_km",
    "effort_score",
    "difficulty",
    "geometry"
]].copy()


points_final.head()


,id,Name,Duration_h,Elevation Gain_m,distance_km,nearest_stop,access_dist_km,effort_score,difficulty,geometry
0,1,Batujajar - Saguling,5.0,79,12.610050,STASIUN CIMINDI,8.216166,3,Moderate,POINT Z (775298.785 9234239.272 656.975)
1,2,Burangrang via Tanjakan Mentari,6.0,932,5.073627,STASIUN CIMINDI,13.463444,6,Hard,POINT Z (782413.7 9250436.797 2091.224)
2,3,Cikole - Parongpong,4.5,367,11.891014,STASIUN CIKUDAPATEUH,15.105966,4,Moderate,POINT Z (793097.106 9249265.548 1315.795)
3,4,Cililin - Saguling,5.0,276,9.643058,STASIUN CIMINDI,13.128927,3,Moderate,POINT Z (771726.661 9230328.961 700.983)
4,5,Citatah - Parahyangan,4.0,287,9.945255,STASIUN CIMINDI,12.810449,3,Moderate,POINT Z (772374.638 9244082.033 731.758)


In [54]:
points_final = points_final.to_crs(epsg=4326)
trails_final = trails_final.to_crs(epsg=4326)

In [ ]:
points_final.to_file("../Data/GeoJSON/points_final.geojson", driver="GeoJSON")
trails_final.to_file("../Data/GeoJSON/trails_final.geojson", driver="GeoJSON")

In [ ]:
print(trails_final.head())
print(points_final.head())
print(len(trails_final), trails_final["id"].nunique())
print(len(points_final), points_final["id"].nunique())

   id                   Name  Duration_h  Elevation Gain_m  distance_km  \
0   1   Batujajar - Saguling         5.0                79    12.610050   
1   3    Cikole - Parongpong         4.5               367    11.891014   
2   4     Cililin - Saguling         5.0               276     9.643058   
3   5  Citatah - Parahyangan         4.0               287     9.945255   
4  12             Papandayan         4.0               687     7.167687   

           nearest_stop  access_dist_km  effort_score difficulty  \
0       STASIUN CIMINDI        8.216166             3   Moderate   
1  STASIUN CIKUDAPATEUH       15.105966             4   Moderate   
2       STASIUN CIMINDI       13.128927             3   Moderate   
3       STASIUN CIMINDI       12.810449             3   Moderate   
4      STASIUN GEDEBAGE       41.537121             5       Hard   

                                            geometry  
0  MULTILINESTRING Z ((107.49131 -6.92121 656.975...  
1  MULTILINESTRING Z ((107.651